In [1]:
import pandas as pd
import sys
sys.path.append('..')

from src.utils import create_player_base, safe_sum, update_rating

In [2]:
import sys
sys.path.append('..')
import pandas as pd
from src.utils import create_player_base, update_rating, safe_sum

In [3]:
all_matches = pd.read_csv('../data/Setka.csv')

# MOV point-based, variable K

In [4]:
all_matches = pd.read_csv('../data/Setka.csv')  # adjust path/filename if different

# Drop rows with missing Sets_P1/Sets_P2 (found in 03 — 1 row caused NaN contamination)
all_matches_clean = all_matches.dropna(subset=['Sets_P1', 'Sets_P2']).reset_index(drop=True)
print(f"Dropped {len(all_matches) - len(all_matches_clean)} row(s) with missing set counts")

Dropped 1 row(s) with missing set counts


In [5]:
def process_matches_variable_k(all_matches, player_base, metric_type="standard",
                                K_new=64, K_veteran=32, veteran_threshold=15):

    results = []

    for match in all_matches.itertuples():
        player_A = match.Player1
        player_B = match.Player2

        r_old_A = player_base.loc[player_base['player'] == player_A, 'rating'].values[0]
        r_old_B = player_base.loc[player_base['player'] == player_B, 'rating'].values[0]

        n_A = player_base.loc[player_base['player'] == player_A, 'matches_played'].values[0]
        n_B = player_base.loc[player_base['player'] == player_B, 'matches_played'].values[0]

        K_A = K_new if n_A < veteran_threshold else K_veteran
        K_B = K_new if n_B < veteran_threshold else K_veteran

        actual_winner = "A" if match.HomeWinner == 1 else "B"
        home_points = safe_sum(match.P1_G1, match.P1_G2, match.P1_G3, match.P1_G4, match.P1_G5)
        away_points = safe_sum(match.P2_G1, match.P2_G2, match.P2_G3, match.P2_G4, match.P2_G5)

        # sets_won/sets_lost must reflect the ACTUAL winner, not P1 vs P2 --
        # Player1 isn't always the winner (see HomeWinner), so this has to
        # be conditional or the set-based multiplier silently breaks.
        if actual_winner == "A":
            sets_won, sets_lost = match.Sets_P1, match.Sets_P2
        else:
            sets_won, sets_lost = match.Sets_P2, match.Sets_P1

        match_data = {
            "sets_won": sets_won,
            "sets_lost": sets_lost,
            "point_diff": abs(home_points - away_points)
        }

        r_new_A, r_new_B, E_A, E_B = update_rating(
            r_old_A, r_old_B, actual_winner, metric_type, match_data, K_A, K_B
        )

        results.append({
            'player1': player_A,
            'player2': player_B,
            'win_probability_p1': E_A,
            'actual_outcome': 1 if actual_winner == "A" else 0,
            'K_A': K_A,
            'K_B': K_B,
        })

        player_base.loc[player_base['player'] == player_A, 'rating'] = r_new_A
        player_base.loc[player_base['player'] == player_B, 'rating'] = r_new_B
        player_base.loc[player_base['player'] == player_A, 'matches_played'] += 1
        player_base.loc[player_base['player'] == player_B, 'matches_played'] += 1

    results_df = pd.DataFrame(results)
    tag = f'{metric_type}_Kvar-new{K_new}-vet{K_veteran}-t{veteran_threshold}'
    results_df.to_csv(f'../results/{tag}_results.csv', index=False)
    player_base.to_csv(f'../data/player_base_{tag}.csv', index=False)

    return results_df, player_base

In [6]:
p1_na_counts = all_matches_clean[['P1_G1','P1_G2','P1_G3','P1_G4','P1_G5']].isna().sum(axis=1)
print(p1_na_counts.value_counts().sort_index())

0    1377
1    1595
2    1299
Name: count, dtype: int64


In [7]:
# Confirm "NA" parsed as real NaN, not the string "NA"
print(all_matches_clean[['P1_G1','P1_G2','P1_G3','P1_G4','P1_G5']].dtypes)

# Check for any match where a player has ALL 5 sets missing (walkover/forfeit edge case)
all_na_rows = all_matches_clean[
    all_matches_clean[['P1_G1','P1_G2','P1_G3','P1_G4','P1_G5']].isna().all(axis=1) |
    all_matches_clean[['P2_G1','P2_G2','P2_G3','P2_G4','P2_G5']].isna().all(axis=1)
]
print(f"Rows with a fully-missing player score: {len(all_na_rows)}")
if len(all_na_rows) > 0:
    display(all_na_rows)  # inspect before deciding whether to drop these too

P1_G1      int64
P1_G2      int64
P1_G3      int64
P1_G4    float64
P1_G5    float64
dtype: object
Rows with a fully-missing player score: 0


In [8]:
player_base = create_player_base(all_matches_clean)

results_df, player_base = process_matches_variable_k(
    all_matches_clean,
    player_base,
    metric_type="points_based",
    K_new=64,
    K_veteran=32,
    veteran_threshold=15
)

In [9]:
print(f"Players with NaN rating: {player_base['rating'].isna().sum()}")  # should be 0
print(f"Rows with NaN win_probability_p1: {results_df['win_probability_p1'].isna().sum()}")  # should be 0

player_base.sort_values('rating', ascending=False).head(10)

Players with NaN rating: 0
Rows with NaN win_probability_p1: 0


,player,rating,matches_played
189,Kolek D.,1737.14,20
195,Polreich T.,1703.52,21
167,Macela P.,1702.40,27
166,Boruvka L.,1702.12,8
288,Zahradka Z.,1694.67,36
212,Volny M.,1692.71,12
101,Fnukal R.,1678.75,38
210,Zobac M.,1665.41,16
155,Mista O.,1661.86,8
123,Paril O.,1661.29,31
